# 📘 Notebook 09: Project: A To-Do List Application

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module B: Mini Projects · Notebook 2 of 3 in this module · (9 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- **Design** a program before writing it: its data, its functions, its user interface
- Represent records as a **list of dictionaries**
- Write one function per operation: add, view, complete, delete, edit
- Guard every operation against invalid input
- **Save** data to a file and **load** it back with the `json` module
- Reflect on design choices, as you would in a short software report

> **Prerequisites:** Module A, especially Notebooks 05 (lists and dictionaries) and 06 (robust programs).
>
> 📝 **About this project.** This notebook follows a real coursework assignment: *design and implement a To-Do List application in Python, then write a short report explaining your design choices.* We will build the application together, stage by stage, and finish with the questions the report should answer.

## 1. Design before code

### Intuition first
An architect does not start by laying bricks. Before writing a single line, we answer three questions on paper.

### Question 1: what should the program do?
The user must be able to:

- add a new task,
- view the existing tasks,
- mark a task as complete,
- delete or rename a task,
- save the tasks to a file and load them back later.

### Question 2: how do we store the data?
Each task has a **name**, a **status** and a **priority**. A dictionary is the natural container for one task, and a list holds them all:

```python
tasks = [
    {"name": "Buy milk", "status": "incomplete", "priority": "low"},
    {"name": "Study loops", "status": "complete", "priority": "high"},
]
```

### Question 3: which functions do we need?
One function per operation:

| Function | What it does |
|---|---|
| `add_task(name, priority)` | adds a task to the list |
| `view_tasks()` | displays all tasks |
| `complete_task(index)` | marks a task as complete |
| `delete_task(index)` | removes a task |
| `edit_task(index, new_name)` | renames a task |
| `save_tasks()` | saves the tasks to a file |
| `load_tasks()` | loads the tasks from a file |

Finally, a **menu loop** lets the user choose an operation. The design is complete. Only now do we start coding.

## 2. Stage 1: adding and viewing tasks

We begin with the two functions that let us see something on the screen. `priority` has a default value, so it may be omitted.

In [ ]:
tasks = []

def add_task(name, priority="low"):
    task = {"name": name, "status": "incomplete", "priority": priority}
    tasks.append(task)
    print(f"Added task: {name} with priority {priority}")

def view_tasks():
    if len(tasks) == 0:
        print("No tasks to show.")
    else:
        for index, task in enumerate(tasks, start=1):
            status = task["status"]
            priority = task["priority"]
            print(f"{index}. {task['name']} - {status} - Priority: {priority}")

# Test the two functions immediately
view_tasks()
add_task("Buy milk")
add_task("Study loops", "high")
add_task("Call Maria", "medium")
view_tasks()

Two details are worth noticing.

- `enumerate(tasks, start=1)` numbers the tasks from **1**. Users count from 1, even though lists count from 0.
- `view_tasks()` handles the **empty list** explicitly. Always ask yourself what your function does when there is nothing to work with.

## 3. Stage 2: completing, deleting and editing

All three functions receive the **index** of a task. An index supplied by a user may be wrong, so each function first checks that it is valid. The condition `0 <= index < len(tasks)` says "the index points to an existing element".

In [ ]:
def complete_task(index):
    if 0 <= index < len(tasks):
        tasks[index]["status"] = "complete"
        print(f"Task {index + 1} marked as complete.")
    else:
        print("Invalid task number.")

def delete_task(index):
    if 0 <= index < len(tasks):
        task_name = tasks[index]["name"]
        del tasks[index]
        print(f"Deleted task: {task_name}")
    else:
        print("Invalid task number.")

def edit_task(index, new_name):
    if 0 <= index < len(tasks):
        old_name = tasks[index]["name"]
        tasks[index]["name"] = new_name
        print(f"Task '{old_name}' has been renamed to '{new_name}'")
    else:
        print("Invalid task number.")

# Test, including an invalid index
complete_task(1)
edit_task(0, "Buy oat milk")
delete_task(2)
complete_task(10)
print()
view_tasks()

> ⚠️ **Common pitfalls**
>
> - The user sees task **1**, but the list stores it at index **0**. Decide in one place where the conversion happens. Here the functions work with list indices, and the menu (Stage 5) subtracts 1 from what the user types.
> - `del tasks[index]` shifts every later task one position to the left, so their numbers change.

## 4. Stage 3: filtering tasks

A useful extra: show only the tasks with a given status. This is the **filter** pattern of Notebook 05, written as a list comprehension.

In [ ]:
def filter_tasks_by_status(status):
    filtered_tasks = [task for task in tasks if task["status"] == status]
    if len(filtered_tasks) == 0:
        print(f"No {status} tasks to show.")
    else:
        for index, task in enumerate(filtered_tasks, start=1):
            print(f"{index}. {task['name']} - Priority: {task['priority']}")

print("Incomplete:")
filter_tasks_by_status("incomplete")
print("Complete:")
filter_tasks_by_status("complete")

## 5. Stage 4: saving and loading

### Why it matters
Everything a program keeps in variables lives in memory, and memory is wiped when the program ends. To make data **persistent**, we must write it to a **file**.

### Working with files
```python
with open("filename", "w") as file:      # "w" = write, "r" = read
    ...                                  # use the file here
```
The `with` statement opens the file and **closes it automatically** at the end of the block, even if an error occurs.

### The JSON format
**JSON** is a text format for storing structured data. It looks almost exactly like Python lists and dictionaries, which makes it ideal for us. The `json` module does the conversion:

- `json.dump(data, file)` writes Python data into a file,
- `json.load(file)` reads it back.

In [ ]:
import json

def save_tasks():
    with open("tasks.json", "w") as file:
        json.dump(tasks, file, indent=4)
    print("Tasks saved to tasks.json.")

def load_tasks():
    global tasks
    try:
        with open("tasks.json", "r") as file:
            tasks = json.load(file)
        print("Tasks loaded from tasks.json.")
    except FileNotFoundError:
        print("No saved tasks found.")
    except json.JSONDecodeError:
        print("Error reading tasks.json.")

save_tasks()

Two points about `load_tasks`.

- `global tasks` is needed because the function **replaces** the whole list with a new one. Without that line, the assignment would create a local variable and the real list would stay unchanged (the scope rule of Notebook 04).
- Loading can fail in two ways: the file may not exist yet, or it may be damaged. We catch each case with its own `except`.

Let us look at what was written, and prove that loading works by emptying the list first.

In [ ]:
with open("tasks.json", "r") as file:
    print(file.read())

tasks = []                 # simulate closing the program: the data is gone
view_tasks()

load_tasks()               # ...and now it is back
view_tasks()

> 🧠 In Google Colab, files are stored in a temporary space that is erased when the session ends. The idea is identical on your own computer, where the file stays on disk.

## 6. Stage 5: the main loop

Every function works and has been tested. The last step is the menu that connects them. Notice how little logic the loop contains: it reads a choice, gathers the information needed, and calls the right function.

In [ ]:
load_tasks()      # load tasks at the beginning, if a file exists

while True:
    print("\nOptions:")
    print("1. Add task")
    print("2. View tasks")
    print("3. Complete task")
    print("4. Delete task")
    print("5. Edit task")
    print("6. Save tasks")
    print("7. Load tasks")
    print("8. Filter by status")
    print("9. Exit")

    choice = input("Choose an option: ")

    if choice == "1":
        task_name = input("Enter task name: ")
        priority = input("Enter task priority (low, medium, high): ").lower()
        add_task(task_name, priority)
    elif choice == "2":
        view_tasks()
    elif choice == "3":
        task_index = int(input("Enter task number to mark complete: ")) - 1
        complete_task(task_index)
    elif choice == "4":
        task_index = int(input("Enter task number to delete: ")) - 1
        delete_task(task_index)
    elif choice == "5":
        task_index = int(input("Enter task number to edit: ")) - 1
        new_name = input("Enter the new task name: ")
        edit_task(task_index, new_name)
    elif choice == "6":
        save_tasks()
    elif choice == "7":
        load_tasks()
    elif choice == "8":
        status = input("Enter status to filter by (incomplete, complete): ").lower()
        filter_tasks_by_status(status)
    elif choice == "9":
        break
    else:
        print("Invalid choice, try again.")

## 7. Reflecting on the design

A working program is half of the assignment. The other half is being able to **explain** it. A short report (about 500 words) should answer three questions. Try to answer them in your own words before reading the notes below.

**1. Software design. How is the application structured?**
The data is a list of dictionaries, one dictionary per task. Each operation is a separate function with one job. The main loop contains no logic of its own, it only connects user choices to functions. This separation means each function could be tested alone, and a new feature needs one new function and one new menu line.

**2. Challenges. What was difficult, and how was it solved?**
Typical answers: the difference between the task number shown to the user and the list index; what happens with an empty list or an invalid number; the need for `global` when loading; a missing file on the first run.

**3. Future improvements. What would you add?**
Deadlines, sorting by priority, searching by keyword, categories, a confirmation before deleting. Several of these are exercises below.

> 🧠 Being able to say *why* your program is built the way it is, and what you would improve, is what separates writing code from **engineering software**.

---
## ✏️ Exercises

Each exercise adds a feature to the application. They assume that the cells above have been run, so that `tasks` and the functions exist.

### Exercise 1 (Count the tasks)
Write a function `count_tasks()` that prints how many tasks are complete and how many are incomplete, for example `2 complete, 3 incomplete`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def count_tasks():
    complete = 0
    incomplete = 0
    for task in tasks:
        if task["status"] == "complete":
            complete += 1
        else:
            incomplete += 1
    print(f"{complete} complete, {incomplete} incomplete")

count_tasks()
```
</details>

### Exercise 2 (Search by keyword)
Write a function `search_tasks(keyword)` that prints every task whose name contains the keyword, ignoring upper and lower case. Print `No matching tasks.` if there are none.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def search_tasks(keyword):
    found = False
    for task in tasks:
        if keyword.lower() in task["name"].lower():
            print(f"- {task['name']} ({task['status']})")
            found = True
    if not found:
        print("No matching tasks.")

add_task("Write the report", "high")
search_tasks("REPORT")
search_tasks("holiday")
```
</details>

### Exercise 3 (Validate the priority)
At the moment `add_task` accepts any text as a priority. Write a new version, `add_task_checked(name, priority)`, that accepts only `low`, `medium` or `high`. For anything else it prints an error and adds nothing.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def add_task_checked(name, priority="low"):
    if priority not in ["low", "medium", "high"]:
        print(f"Invalid priority: {priority}")
        return
    add_task(name, priority)

add_task_checked("Water the plants", "urgent")
add_task_checked("Water the plants", "medium")
```
</details>

### Exercise 4 (Show by priority)
Write a function `view_by_priority()` that shows the tasks grouped by priority: first all the `high` ones, then `medium`, then `low`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def view_by_priority():
    for level in ["high", "medium", "low"]:
        print(f"{level.upper()}:")
        for task in tasks:
            if task["priority"] == level:
                print(f"  - {task['name']} ({task['status']})")

view_by_priority()
```
</details>

### Exercise 5 (Clear completed tasks)
Write a function `clear_completed()` that removes all complete tasks from the list and prints how many were removed. (Hint: it is risky to delete from a list while looping over it. Build a new list with the tasks you want to **keep** instead.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def clear_completed():
    global tasks
    before = len(tasks)
    tasks = [task for task in tasks if task["status"] != "complete"]
    print(f"Removed {before - len(tasks)} completed tasks.")

complete_task(0)
clear_completed()
view_tasks()
```

*Deleting elements while looping over the same list makes the loop skip items, because the remaining elements shift left. Building a new list avoids the problem entirely.*
</details>

### Exercise 6 (A task with a deadline, a little harder)
Write a function `add_task_with_deadline(name, priority, deadline)` that stores a fourth key, `deadline`, as text in the form `YYYY-MM-DD`. Then write `view_by_deadline()`, which prints the tasks that have a deadline, earliest first. (Hint: dates written as `YYYY-MM-DD` sort correctly as plain text. `sorted(list_of_dicts, key=lambda t: t["deadline"])` sorts dictionaries by one of their values.)

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def add_task_with_deadline(name, priority, deadline):
    task = {"name": name, "status": "incomplete", "priority": priority, "deadline": deadline}
    tasks.append(task)

def view_by_deadline():
    with_deadline = [task for task in tasks if "deadline" in task]
    for task in sorted(with_deadline, key=lambda t: t["deadline"]):
        print(f"{task['deadline']}  {task['name']}")

add_task_with_deadline("Submit assignment", "high", "2026-12-01")
add_task_with_deadline("Book tickets", "low", "2026-11-15")
view_by_deadline()
```

*A `lambda` is a tiny function written in one line. `lambda t: t["deadline"]` means "given a task `t`, use its deadline as the value to sort by". Sorting is the subject of Notebook 14.*
</details>

## 🔑 Key takeaways

- **Design before code**: decide what the program does, how the data is stored, and which functions are needed.
- A **list of dictionaries** is the natural way to store a collection of records.
- Write **one function per operation**, and test each one as soon as it is written.
- Check every index that comes from the user: `0 <= index < len(tasks)`.
- Data in variables disappears when the program ends; **files** make it persistent, and `json` stores lists and dictionaries directly.
- A good main loop contains almost no logic: it only connects choices to functions.
- Be ready to **explain** your design, its difficulties, and its possible improvements.

---
**Next:** *Notebook 10: An ATM as a State Machine*, a different way to organise an interactive program.

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*